<a href="https://colab.research.google.com/github/Divya6859/imdb-sentiment-analysis/blob/main/IMDb_Movie_Review_Scrapping_%2B_Sentiment_Analysis_using_NLP_and_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install playwright
!playwright install-deps
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 20.0 MB/s eta 0:00:00
Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,978 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,844 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,960 kB]
Get:13 htt

In [2]:
import asyncio
from playwright.async_api import async_playwright

async def run_scraper():
    async with async_playwright() as p:
        # Launch the browser
        browser = await p.chromium.launch(headless=True)

        # Set a standard user agent to avoid basic blocks
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )

        page = await context.new_page()

        print("Navigating to IMDb...")
        await page.goto('https://www.imdb.com/title/tt15097216/reviews/', wait_until="domcontentloaded")

        # Wait for the review elements to load into the DOM
        try:
            await page.wait_for_selector('.ipc-html-content-inner-div', timeout=10000)

            # Extract the text from all matching elements
            reviews = await page.locator('.ipc-html-content-inner-div').all_inner_texts()

            if reviews:
                with open('imdb_reviews.txt', 'w', encoding='utf-8') as f:
                    f.write("\n\n".join(reviews))
                print(f"Success! Saved {len(reviews)} reviews to imdb_reviews.txt")
            else:
                print("Found the elements, but they contained no text.")

        except Exception as e:
            print(f"Timeout or error: {e}")

        await browser.close()

# In a Notebook, you can't use asyncio.run().
# Instead, use 'await' directly or use the existing loop:
await run_scraper()

Navigating to IMDb...
Success! Saved 21 reviews to imdb_reviews.txt


In [3]:
import csv

# Read the reviews from the text file created in the previous step
try:
    with open('imdb_reviews.txt', 'r', encoding='utf-8') as f:
        # Assuming reviews are separated by double newlines as per the previous cell's logic
        reviews = f.read().split('\n\n')

    # Filter out any empty strings
    reviews = [r.strip() for r in reviews if r.strip()]

    if reviews:
        # Save to CSV
        csv_filename = 'imdb_reviews.csv'
        with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Review Text'])
            for review in reviews:
                writer.writerow([review])

        print(f"Successfully saved {len(reviews)} reviews to {csv_filename}")
    else:
        print("No reviews found in the source file.")
except FileNotFoundError:
    print("Error: 'imdb_reviews.txt' not found. Please run the scraper cell first.")

Successfully saved 59 reviews to imdb_reviews.csv


**IMDb 50K Movie Reviews Dataset**

1. Import Libraries

In [4]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, confusion_matrix, f1_score

2. Load Dataset

In [6]:
df = pd.read_csv('/content/IMDB Dataset.csv', on_bad_lines='skip', engine='python')
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


3. Convert Labels

In [7]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

4. Text Preprocessing

In [8]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_review'] = df['review'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


5. Train-Test Split

In [9]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

6. TF-IDF

In [10]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

7. MODEL BUILDING

Logistic Regression

In [11]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)

Naive Bayes

In [12]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
nb_pred = nb.predict(X_test_tfidf)

Random Forest

In [13]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_tfidf, y_train)
rf_pred = rf.predict(X_test_tfidf)

8. EVALUATION FUNCTION

In [14]:
def evaluate(name, y_test, y_pred):
    print(f"\n{name}")
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

9. MODEL COMPARISON TABLE

In [15]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Random Forest"],
    "F1 Score": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, nb_pred),
        f1_score(y_test, rf_pred)
    ]
})

print(results)

                 Model  F1 Score
0  Logistic Regression  0.893617
1          Naive Bayes  0.856549
2        Random Forest  0.852796


10. FINAL MODEL SELECTION

In [16]:
best_model = lr  # or nb or rf based on results

11. PREDICTION SYSTEM

In [17]:
def predict_review(text):
    cleaned = clean_text(text)
    vector = tfidf.transform([cleaned])
    pred = best_model.predict(vector)[0]

    if pred == 1:
        print("Positive Review 😊")
    else:
        print("Negative Review 😞")

predict_review("This movie was absolutely amazing and fantastic!")
predict_review("Worst movie ever, very boring and bad")

Positive Review 😊
Negative Review 😞


In [18]:
import joblib

joblib.dump(best_model, "model.pkl")
joblib.dump(tfidf, "tfidf.pkl")

['tfidf.pkl']